# 08 — Leakage-Safe Feature Assembly

Yeni inference yapılmaz. Frozen artifact'lar birleştirilir:

```text
04 internal features
+ 05 self-confidence / self-verification
+ 06 V2 self-consistency
+ 07 manual correctness
→ leakage-safe feature table
```

Çıktılar:

```text
features/assembled/validated_v002/
├── master_feature_table_v1.parquet
├── core_model_matrix_v1.parquet
├── extended_model_matrix_v1.parquet
├── feature_schema_v1.json
└── stage08_manifest_v1.json
```

`manual_is_correct` yalnız target'tır. Reference answer, aliases, raw answer text, token arrays,
`manual_error_type` ve gold-derived alanlar predictor listesine girmez.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install "pyarrow>=15,<25"

Mounted at /content/drive


In [2]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

VALIDATED_DATASET_VERSION = "validated_v012_final250_v1"
FEATURE_ASSEMBLY_VERSION = "feature_assembly_v1"
FEATURE_SCHEMA_VERSION = "feature_schema_v1"

R04 = PROJECT_ROOT / "results" / "stage04"
R05 = PROJECT_ROOT / "results" / "stage05"
R06 = PROJECT_ROOT / "results" / "stage06"
R07 = PROJECT_ROOT / "results" / "stage07"
R08 = PROJECT_ROOT / "results" / "stage08"

OUTPUT_DIR = (
    PROJECT_ROOT / "features" / "assembled" / VALIDATED_DATASET_VERSION
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
R08.mkdir(parents=True, exist_ok=True)

P04 = R04 / f"{VALIDATED_DATASET_VERSION}_stage04_complete.json"
P05 = R05 / f"{VALIDATED_DATASET_VERSION}_stage05_complete.json"
P06 = R06 / f"{VALIDATED_DATASET_VERSION}_stage06_v2_complete.json"
P07 = R07 / f"{VALIDATED_DATASET_VERSION}_stage07_complete.json"

MASTER_PATH = OUTPUT_DIR / "master_feature_table_v1.parquet"
CORE_PATH = OUTPUT_DIR / "core_model_matrix_v1.parquet"
EXTENDED_PATH = OUTPUT_DIR / "extended_model_matrix_v1.parquet"
SCHEMA_PATH = OUTPUT_DIR / "feature_schema_v1.json"
MANIFEST_PATH = OUTPUT_DIR / "stage08_manifest_v1.json"

JOIN_AUDIT_PATH = R08 / f"{VALIDATED_DATASET_VERSION}_join_audit_v1.csv"
FEATURE_AUDIT_PATH = R08 / f"{VALIDATED_DATASET_VERSION}_feature_audit_v1.csv"
LABEL_AUDIT_PATH = R08 / f"{VALIDATED_DATASET_VERSION}_label_audit_v1.csv"

In [3]:
def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def canonical_json(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def sha256_object(obj):
    return sha256_text(canonical_json(obj))

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def require(path):
    if not Path(path).exists():
        raise FileNotFoundError(path)

def assert_unique(df, name):
    if df.duplicated(["question_id", "model_id"]).any():
        raise RuntimeError(f"{name}: duplicate question_id × model_id")

def key_set(df):
    return set(zip(df["question_id"], df["model_id"]))

## 1. Upstream manifest chain

In [4]:
for p in [P04, P05, P06, P07]:
    require(p)

m04 = load_json(P04)
m05 = load_json(P05)
m06 = load_json(P06)
m07 = load_json(P07)

for name, m in [("04", m04), ("05", m05), ("06", m06), ("07", m07)]:
    if m["dataset_version"] != VALIDATED_DATASET_VERSION:
        raise RuntimeError(f"Stage{name} dataset version mismatch")

sha04 = file_sha256(P04)
sha05 = file_sha256(P05)
sha06 = file_sha256(P06)
sha07 = file_sha256(P07)

if m06["stage04_completion_manifest_sha256"] != sha04:
    raise RuntimeError("Stage06 -> Stage04 SHA chain mismatch")

if m06["stage05_completion_manifest_sha256"] != sha05:
    raise RuntimeError("Stage06 -> Stage05 SHA chain mismatch")

if m07["stage04_completion_sha256"] != sha04:
    raise RuntimeError("Stage07 -> Stage04 SHA chain mismatch")

if m07["stage06_v2_completion_sha256"] != sha06:
    raise RuntimeError("Stage07 -> Stage06 SHA chain mismatch")

NQ = int(m04["n_questions"])
NM = int(m04["n_models"])
EXPECTED = NQ * NM

print("Questions:", NQ)
print("Models:", NM)
print("Expected rows:", EXPECTED)
print("UPSTREAM MANIFEST CHAIN: PASS")

Questions: 1250
Models: 3
Expected rows: 3750
UPSTREAM MANIFEST CHAIN: PASS


## 2. Stage-04 yükle

In [5]:
frames = []

for artifact in m04["model_exports"]:
    p = Path(artifact["export_file"])
    require(p)

    if file_sha256(p) != artifact["export_sha256"]:
        raise RuntimeError(f"Stage04 export SHA mismatch: {artifact['model_id']}")

    f = pd.DataFrame(read_jsonl(p))

    if len(f) != NQ:
        raise RuntimeError(f"Stage04 N mismatch: {artifact['model_id']}")

    frames.append(f)

df04 = pd.concat(frames, ignore_index=True)
assert_unique(df04, "Stage04")

if len(df04) != EXPECTED:
    raise RuntimeError("Stage04 total N mismatch")

df04["stage04_inference_cache_key"] = df04["inference_cache_key"]
df04["model_answer_sha256"] = df04["answer"].astype(str).map(sha256_text)

print("Stage04 rows:", len(df04))

Stage04 rows: 3750


## 3. Stage-05 yükle

In [6]:
conf_frames = []
ver_frames = []

for artifact in m05["model_artifacts"]:
    cp = Path(artifact["self_confidence_export"])
    vp = Path(artifact["self_verification_export"])

    require(cp)
    require(vp)

    if file_sha256(cp) != artifact["self_confidence_export_sha256"]:
        raise RuntimeError(f"Stage05 confidence SHA mismatch: {artifact['model_id']}")

    if file_sha256(vp) != artifact["self_verification_export_sha256"]:
        raise RuntimeError(f"Stage05 verification SHA mismatch: {artifact['model_id']}")

    conf_frames.append(pd.DataFrame(read_jsonl(cp)))
    ver_frames.append(pd.DataFrame(read_jsonl(vp)))

df05c = pd.concat(conf_frames, ignore_index=True)
df05v = pd.concat(ver_frames, ignore_index=True)

for df, name in [(df05c, "Stage05 confidence"), (df05v, "Stage05 verification")]:
    assert_unique(df, name)
    if len(df) != EXPECTED:
        raise RuntimeError(f"{name}: row count mismatch")

print("Confidence:", len(df05c))
print("Verification:", len(df05v))

Confidence: 3750
Verification: 3750


## 4. Stage-06 V2 ve Stage-07 labels yükle

In [7]:
frames = []

for artifact in m06["model_artifacts"]:
    p = Path(artifact["export_file"])
    require(p)

    if file_sha256(p) != artifact["export_sha256"]:
        raise RuntimeError(f"Stage06 SHA mismatch: {artifact['model_id']}")

    frames.append(pd.DataFrame(read_jsonl(p)))

df06 = pd.concat(frames, ignore_index=True)
assert_unique(df06, "Stage06")

if len(df06) != EXPECTED:
    raise RuntimeError("Stage06 row count mismatch")

labels_path = Path(m07["final_labels_file"])
require(labels_path)

if file_sha256(labels_path) != m07["final_labels_sha256"]:
    raise RuntimeError("Stage07 final labels SHA mismatch")

df07 = pd.DataFrame(read_jsonl(labels_path))
assert_unique(df07, "Stage07")

if len(df07) != EXPECTED:
    raise RuntimeError("Stage07 row count mismatch")

if not set(df07["manual_is_correct"].unique()).issubset({0, 1}):
    raise RuntimeError("manual_is_correct binary değil")

print("Stage06:", len(df06))
print("Stage07:", len(df07))
print("Canonicalization:", m06["canonicalization_version"])

Stage06: 3750
Stage07: 3750
Canonicalization: answer_type_canonicalization_tr_v2


## 5. Cross-stage key ve frozen-answer SHA audit

In [8]:
base_keys = key_set(df04)

audit_rows = []

for name, df in [
    ("stage05_confidence", df05c),
    ("stage05_verification", df05v),
    ("stage06_v2", df06),
    ("stage07_labels", df07),
]:
    keys = key_set(df)

    audit_rows.append({
        "artifact": name,
        "n_rows": len(df),
        "n_unique_keys": len(keys),
        "missing_stage04_keys": len(base_keys - keys),
        "extra_keys": len(keys - base_keys),
    })

    if keys != base_keys:
        raise RuntimeError(f"Key-set mismatch: {name}")

join_audit = pd.DataFrame(audit_rows)
display(join_audit)
join_audit.to_csv(JOIN_AUDIT_PATH, index=False)

base = df04.set_index(["question_id", "model_id"])

def validate_link(df, name, answer_hash_col):
    other = df.set_index(["question_id", "model_id"])

    for key in base.index:
        b = base.loc[key]
        o = other.loc[key]

        if str(b["model_revision"]) != str(o["model_revision"]):
            raise RuntimeError(f"{name}: model_revision mismatch at {key}")

        if str(b["stage04_inference_cache_key"]) != str(o["stage04_inference_cache_key"]):
            raise RuntimeError(f"{name}: Stage04 cache-key mismatch at {key}")

        if str(b["model_answer_sha256"]) != str(o[answer_hash_col]):
            raise RuntimeError(f"{name}: frozen answer SHA mismatch at {key}")

validate_link(df05c, "Stage05 confidence", "frozen_answer_sha256")
validate_link(df05v, "Stage05 verification", "frozen_answer_sha256")
validate_link(df06, "Stage06 V2", "frozen_deterministic_answer_sha256")
validate_link(df07, "Stage07 labels", "model_answer_sha256")

print("CROSS-STAGE LINKAGE: PASS")

,artifact,n_rows,n_unique_keys,missing_stage04_keys,extra_keys
0,stage05_confidence,3750,3750,0,0
1,stage05_verification,3750,3750,0,0
2,stage06_v2,3750,3750,0,0
3,stage07_labels,3750,3750,0,0


CROSS-STAGE LINKAGE: PASS


## 6. Scalar leakage-safe merge

In [9]:
keep04 = [
    "dataset_version",
    "question_id",
    "model_id",
    "model_short_name",
    "model_revision",
    "stage04_inference_cache_key",
    "model_answer_sha256",
    "answer_type",
    "category",
    "used_in_model_selection",
    "generation_seconds",
    "generated_token_count_total",
    "output_content_token_count",
    "output_char_count",
    "sequence_logprob_sum",
    "mean_logprob",
    "min_logprob",
    "first_token_logprob",
    "mean_token_entropy",
    "mean_token_entropy_normalized",
    "max_token_entropy_normalized",
    "first_token_entropy_normalized",
    "pure_unknown_response",
    "contains_unknown_marker",
    "mixed_answer_unknown",
    "refusal_indicator",
    "empty_response",
    "ended_with_eos",
    "truncated_by_max_tokens",
]

keep05c = [
    "question_id",
    "model_id",
    "self_confidence",
    "self_confidence_scaled",
    "self_confidence_parse_success",
    "self_confidence_format_valid",
    "self_confidence_generation_seconds",
]

keep05v = [
    "question_id",
    "model_id",
    "verification_choice",
    "verification_parse_success",
    "verification_format_valid",
    "verification_p_correct",
    "verification_p_incorrect",
    "verification_p_uncertain",
    "verification_margin",
    "verification_entropy_normalized",
    "verification_argmax_class",
    "verification_generation_probability_agreement",
    "verification_generation_seconds",
]

keep06 = [
    "question_id",
    "model_id",
    "K",
    "self_consistency",
    "majority_count",
    "unique_answer_count",
    "unique_answer_ratio",
    "sample_answer_entropy_normalized",
    "deterministic_majority_agreement",
    "deterministic_sample_agreement_rate",
    "unknown_sample_rate",
    "contains_unknown_sample_rate",
    "mixed_unknown_sample_rate",
    "refusal_sample_rate",
    "empty_sample_rate",
    "sample_truncation_rate",
    "surface_self_consistency",
    "surface_unique_answer_count",
    "surface_unique_answer_ratio",
    "surface_sample_answer_entropy_normalized",
    "canonicalization_changed_sample_count",
    "canonicalization_reduced_unique_count",
]

keep07 = [
    "question_id",
    "model_id",
    "manual_is_correct",
    "manual_error_type",
]

master = (
    df04[keep04]
    .merge(df05c[keep05c], on=["question_id", "model_id"], validate="one_to_one")
    .merge(df05v[keep05v], on=["question_id", "model_id"], validate="one_to_one")
    .merge(df06[keep06], on=["question_id", "model_id"], validate="one_to_one")
    .merge(df07[keep07], on=["question_id", "model_id"], validate="one_to_one")
)

if len(master) != EXPECTED:
    raise RuntimeError("Master row count mismatch")

assert_unique(master, "Master")

master["question_group_id"] = master["question_id"]
master["exclude_from_final_test"] = master["used_in_model_selection"].astype(int)
master["unknown_response"] = master["pure_unknown_response"].astype(int)

print("Master rows:", len(master))
print("Master columns:", len(master.columns))

Master rows: 3750
Master columns: 70


## 7. Predictor schema

In [10]:
TARGET = "manual_is_correct"

CORE_FEATURES = [
    "mean_logprob",
    "min_logprob",
    "mean_token_entropy_normalized",
    "output_content_token_count",
    "self_confidence_scaled",
    "verification_p_correct",
    "self_consistency",
    "unknown_response",
]

EXTENDED_ONLY_FEATURES = [
    "first_token_logprob",
    "max_token_entropy_normalized",
    "first_token_entropy_normalized",
    "verification_p_uncertain",
    "verification_margin",
    "verification_entropy_normalized",
    "unique_answer_ratio",
    "sample_answer_entropy_normalized",
    "deterministic_majority_agreement",
    "deterministic_sample_agreement_rate",
    "contains_unknown_marker",
    "mixed_answer_unknown",
    "refusal_indicator",
    "unknown_sample_rate",
    "contains_unknown_sample_rate",
    "mixed_unknown_sample_rate",
    "refusal_sample_rate",
    "truncated_by_max_tokens",
    "sample_truncation_rate",
]

EXTENDED_FEATURES = CORE_FEATURES + EXTENDED_ONLY_FEATURES

PROHIBITED = [
    "reference_answer",
    "accepted_aliases",
    "automatic_alias_match",
    "sample_correct_rate",
    "annotation_notes",
    "question",
    "answer",
    "content_token_ids",
    "content_token_strings",
    "token_logprobs",
    "token_entropies",
    "token_entropies_normalized",
]

LABEL_METADATA = ["manual_error_type"]

if TARGET in CORE_FEATURES or TARGET in EXTENDED_FEATURES:
    raise RuntimeError("Target predictor listesinde")

if "manual_error_type" in CORE_FEATURES or "manual_error_type" in EXTENDED_FEATURES:
    raise RuntimeError("manual_error_type predictor listesinde")

missing_features = [
    f for f in EXTENDED_FEATURES if f not in master.columns
]

if missing_features:
    raise RuntimeError(f"Missing predictor fields: {missing_features}")

present_prohibited = [c for c in PROHIBITED if c in master.columns]

if present_prohibited:
    raise RuntimeError(f"Prohibited leakage fields in master: {present_prohibited}")

print("CORE:", len(CORE_FEATURES), CORE_FEATURES)
print("EXTENDED:", len(EXTENDED_FEATURES))

CORE: 8 ['mean_logprob', 'min_logprob', 'mean_token_entropy_normalized', 'output_content_token_count', 'self_confidence_scaled', 'verification_p_correct', 'self_consistency', 'unknown_response']
EXTENDED: 27


## Hatalı Aşamalar Neler?

In [11]:

import numpy as np
import pandas as pd

# Eksik self-confidence değerlerini tespit et
confidence_values = pd.to_numeric(
    master["self_confidence_scaled"],
    errors="coerce"
)

invalid_mask = ~np.isfinite(confidence_values.to_numpy())

problematic_rows = master.loc[
    invalid_mask,
    [
        "question_id",
        "model_id",
        "self_confidence",
        "self_confidence_scaled",
        "self_confidence_parse_success",
        "self_confidence_format_valid",
        "manual_is_correct",
    ],
].copy()

print("Problemli kayıt sayısı:", len(problematic_rows))

display(problematic_rows)

# Aynı kayıtların Stage05 kaynak verisini incele
problematic_keys = problematic_rows[
    ["question_id", "model_id"]
]

source_rows = df05c.merge(
    problematic_keys,
    on=["question_id", "model_id"],
    how="inner",
    validate="one_to_one",
)

print("\nStage05 kaynak kayıtları:")

for _, row in source_rows.iterrows():
    print("\n" + "=" * 60)

    for column, value in row.items():
        print(f"{column}: {value}")

Problemli kayıt sayısı: 2


,question_id,model_id,self_confidence,self_confidence_scaled,self_confidence_parse_success,self_confidence_format_valid,manual_is_correct
2130,TRQA000881,Qwen/Qwen3-4B,NaN,NaN,0,0,0
2869,TRQA000370,microsoft/Phi-4-mini-instruct,NaN,NaN,0,0,0



Stage05 kaynak kayıtları:

dataset_version: validated_v012_final250_v1
question_id: TRQA000881
model_id: Qwen/Qwen3-4B
model_revision: 3101254bbe4169895668a0e7653c3fd1f313576e
stage04_inference_cache_key: 40c486ec54134123ac558e5d793dcb61abb9374238766b81167d7f601b123047
frozen_answer_sha256: 6b39fc4853e4e764ccafd7f0c0edf373e4570ee5bf9802a67223f79ba05119d4
self_confidence_raw: 400
self_confidence: nan
self_confidence_scaled: nan
self_confidence_parse_success: 0
self_confidence_format_valid: 0
self_confidence_generation_seconds: 0.4145419500000571

dataset_version: validated_v012_final250_v1
question_id: TRQA000370
model_id: microsoft/Phi-4-mini-instruct
model_revision: cfbefacb99257ffa30c83adab238a50856ac3083
stage04_inference_cache_key: e9b420ea062bf80b557afbeaa8929ea5880946e6110511b2fe963316c7b05abb
frozen_answer_sha256: 5253d78a5a4837b01dd0b6b4c255ba7456be26f35f061d345acc91ca5449b7d2
self_confidence_raw: İlk cevabı kontrol ett
self_confidence: nan
self_confidence_scaled: nan
self_con

## 8. Numeric integrity ve class-balance audit

In [12]:
audit_rows = []

for model_id, model_df in master.groupby("model_id"):
    for feature in EXTENDED_FEATURES:
        values = pd.to_numeric(model_df[feature], errors="coerce")
        arr = values.to_numpy(dtype=float)

        audit_rows.append({
            "model_id": model_id,
            "feature": feature,
            "n": len(model_df),
            "missing_count": int(values.isna().sum()),
            "nonfinite_count": int((~np.isfinite(arr)).sum()),
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
        })

feature_audit = pd.DataFrame(audit_rows)

if feature_audit["missing_count"].sum() != 0:
    display(feature_audit[feature_audit["missing_count"] > 0])
    raise RuntimeError("Predictor missing values bulundu")

if feature_audit["nonfinite_count"].sum() != 0:
    display(feature_audit[feature_audit["nonfinite_count"] > 0])
    raise RuntimeError("Predictor non-finite values bulundu")

feature_audit.to_csv(FEATURE_AUDIT_PATH, index=False)

label_audit = (
    master.groupby("model_id")
    .agg(
        n=("question_id", "count"),
        n_correct=(TARGET, "sum"),
        accuracy=(TARGET, "mean"),
        n_incorrect=(TARGET, lambda s: int((s == 0).sum())),
        model_selection_overlap_n=("used_in_model_selection", "sum"),
    )
    .reset_index()
)

display(label_audit)
label_audit.to_csv(LABEL_AUDIT_PATH, index=False)

if not (label_audit["n"] == NQ).all():
    raise RuntimeError("Per-model coverage mismatch")

print("LEAKAGE + NUMERIC INTEGRITY: PASS")

,model_id,feature,n,missing_count,nonfinite_count,min,max,mean
4,Qwen/Qwen3-4B,self_confidence_scaled,1250,1,1,0.0,1.0,0.751137
58,microsoft/Phi-4-mini-instruct,self_confidence_scaled,1250,1,1,0.0,1.0,0.715132


RuntimeError: Predictor missing values bulundu

## 9. Core / Extended model matrices

In [13]:
MATRIX_METADATA = [
    "dataset_version",
    "question_id",
    "question_group_id",
    "model_id",
    "model_revision",
    "answer_type",
    "category",
    "used_in_model_selection",
    "exclude_from_final_test",
]

core_columns = MATRIX_METADATA + CORE_FEATURES + [TARGET]
extended_columns = MATRIX_METADATA + EXTENDED_FEATURES + [TARGET]

core_df = master[core_columns].copy()
extended_df = master[extended_columns].copy()

for forbidden in PROHIBITED + LABEL_METADATA:
    if forbidden in core_df.columns or forbidden in extended_df.columns:
        raise RuntimeError(f"Model matrix leakage: {forbidden}")

print("Core matrix:", core_df.shape)
print("Extended matrix:", extended_df.shape)

Core matrix: (3750, 18)
Extended matrix: (3750, 37)


## 10. Feature schema

In [14]:
schema_body = {
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "assembly_version": FEATURE_ASSEMBLY_VERSION,
    "dataset_version": VALIDATED_DATASET_VERSION,
    "row_unit": "one frozen model answer for one question",
    "primary_key": ["question_id", "model_id"],
    "group_key_for_cross_validation": "question_group_id",
    "target": TARGET,
    "core_features": CORE_FEATURES,
    "extended_only_features": EXTENDED_ONLY_FEATURES,
    "extended_features": EXTENDED_FEATURES,
    "prohibited_leakage_columns": PROHIBITED,
    "label_metadata_columns": LABEL_METADATA,
    "final_test_policy": {
        "final_test_split_created_here": False,
        "used_in_model_selection_must_be_excluded": True,
        "exclusion_column": "exclude_from_final_test",
    },
    "modeling_policy": {
        "default": "separate post-hoc calibrator per model",
        "pooled_analysis_requires_question_grouped_splits": True,
        "answer_type_default_predictor": False,
        "category_default_predictor": False,
    },
    "self_consistency": {
        "version": m06["self_consistency_version"],
        "canonicalization_version": m06["canonicalization_version"],
        "K": m06["K"],
    },
}

schema_hash = sha256_object(schema_body)
schema = {**schema_body, "schema_sha256": schema_hash}

if SCHEMA_PATH.exists():
    existing = load_json(SCHEMA_PATH)
    if existing["schema_sha256"] != schema_hash:
        raise RuntimeError("Existing feature schema differs")
else:
    with open(SCHEMA_PATH, "x", encoding="utf-8") as f:
        json.dump(schema, f, ensure_ascii=False, indent=2)

print("Feature schema SHA:", schema_hash)
print(SCHEMA_PATH)

Feature schema SHA: 3b003982fd39ad9e1e2ba22e9761c3c677dc8eb527972581609c47b9c2acc9a9
/content/drive/MyDrive/LLM_Confidence_Thesis/features/assembled/validated_v012_final250_v1/feature_schema_v1.json


## 11. Parquet artifact'ları yaz

In [15]:
def write_or_verify(df, path):
    if path.exists():
        existing = pd.read_parquet(path)

        if list(existing.columns) != list(df.columns):
            raise RuntimeError(f"Existing columns differ: {path}")

        if key_set(existing) != key_set(df):
            raise RuntimeError(f"Existing key set differs: {path}")

        print("Already exists:", path)
        return

    df.to_parquet(path, index=False, engine="pyarrow")
    print("Created:", path)

write_or_verify(master, MASTER_PATH)
write_or_verify(core_df, CORE_PATH)
write_or_verify(extended_df, EXTENDED_PATH)

for p in [MASTER_PATH, CORE_PATH, EXTENDED_PATH]:
    reloaded = pd.read_parquet(p)

    if len(reloaded) != EXPECTED:
        raise RuntimeError(f"Reload row count mismatch: {p}")

    assert_unique(reloaded, p.name)

print("PARQUET RELOAD + KEY SANITY: PASS")
print("Master SHA:", file_sha256(MASTER_PATH))
print("Core SHA:", file_sha256(CORE_PATH))
print("Extended SHA:", file_sha256(EXTENDED_PATH))

Created: /content/drive/MyDrive/LLM_Confidence_Thesis/features/assembled/validated_v012_final250_v1/master_feature_table_v1.parquet
Created: /content/drive/MyDrive/LLM_Confidence_Thesis/features/assembled/validated_v012_final250_v1/core_model_matrix_v1.parquet
Created: /content/drive/MyDrive/LLM_Confidence_Thesis/features/assembled/validated_v012_final250_v1/extended_model_matrix_v1.parquet
PARQUET RELOAD + KEY SANITY: PASS
Master SHA: 3d29e1d92fabc4b60b6d0ec5ba96a9d1bd2931a91dd5bfbfd12e58f874d558de
Core SHA: 9003bcc9c4e11fa1c4d68259fad5420e86c7919caa6a5a04670f5215320209f0
Extended SHA: 3d6f6868a0ab9a146aaf066bd35ab306d13132537fd0ccdaae434811d0f34474


## 12. Preview

In [16]:
preview_columns = [
    "question_id",
    "model_id",
    "answer_type",
    "mean_logprob",
    "mean_token_entropy_normalized",
    "self_confidence_scaled",
    "verification_p_correct",
    "self_consistency",
    "unknown_response",
    "manual_is_correct",
]

display(
    master[preview_columns]
    .sort_values(["question_id", "model_id"])
    .head(18)
)

,question_id,model_id,answer_type,mean_logprob,mean_token_entropy_normalized,self_confidence_scaled,verification_p_correct,self_consistency,unknown_response,manual_is_correct
1250,TRQA000001,Qwen/Qwen3-4B,ENTITY_PERSON,-0.665925,0.143501,0.00,1.020345e-09,0.2,0,0
0,TRQA000001,google/gemma-3-4b-it,ENTITY_PERSON,-0.276380,0.048207,0.95,9.998410e-01,0.2,0,0
2500,TRQA000001,microsoft/Phi-4-mini-instruct,ENTITY_PERSON,-0.836065,0.175471,1.00,9.789540e-01,0.2,0,0
1251,TRQA000002,Qwen/Qwen3-4B,ENTITY_PERSON,-0.356584,0.082444,0.70,3.726618e-06,0.4,0,0
1,TRQA000002,google/gemma-3-4b-it,ENTITY_PERSON,-0.023677,0.010245,1.00,9.999960e-01,1.0,0,0
2501,TRQA000002,microsoft/Phi-4-mini-instruct,ENTITY_PERSON,-0.208254,0.062680,1.00,9.914160e-01,1.0,0,0
1252,TRQA000003,Qwen/Qwen3-4B,ENTITY_PERSON,-0.007067,0.003445,1.00,1.000000e+00,1.0,0,1
2,TRQA000003,google/gemma-3-4b-it,ENTITY_PERSON,-0.002199,0.001103,0.95,9.999985e-01,1.0,0,1
2502,TRQA000003,microsoft/Phi-4-mini-instruct,ENTITY_PERSON,-0.031884,0.013014,1.00,9.960503e-01,1.0,0,1
1253,TRQA000004,Qwen/Qwen3-4B,ENTITY_PERSON,-0.230125,0.047323,1.00,9.999999e-01,0.8,0,1


## 13. Stage-08 manifest

In [17]:
manifest_body = {
    "stage": "08_feature_assembly",
    "assembly_version": FEATURE_ASSEMBLY_VERSION,
    "feature_schema_version": FEATURE_SCHEMA_VERSION,
    "feature_schema_sha256": schema_hash,
    "dataset_version": VALIDATED_DATASET_VERSION,
    "n_questions": NQ,
    "n_models": NM,
    "n_model_answer_rows": len(master),
    "target": TARGET,
    "core_features": CORE_FEATURES,
    "extended_features": EXTENDED_FEATURES,
    "upstream": {
        "stage04_completion_sha256": sha04,
        "stage05_completion_sha256": sha05,
        "stage06_v2_completion_sha256": sha06,
        "stage07_completion_sha256": sha07,
    },
    "outputs": {
        "master_feature_table": str(MASTER_PATH),
        "master_feature_table_sha256": file_sha256(MASTER_PATH),
        "core_model_matrix": str(CORE_PATH),
        "core_model_matrix_sha256": file_sha256(CORE_PATH),
        "extended_model_matrix": str(EXTENDED_PATH),
        "extended_model_matrix_sha256": file_sha256(EXTENDED_PATH),
        "feature_schema": str(SCHEMA_PATH),
        "feature_schema_file_sha256": file_sha256(SCHEMA_PATH),
    },
    "audits": {
        "join_audit": str(JOIN_AUDIT_PATH),
        "feature_audit": str(FEATURE_AUDIT_PATH),
        "label_audit": str(LABEL_AUDIT_PATH),
    },
    "question_grouping_required_for_pooled_cv": True,
    "final_test_split_created": False,
    "model_selection_overlap_excluded_from_future_final_test": True,
    "created_at_utc": utc_now_iso(),
}

manifest_hash = sha256_object(manifest_body)
manifest = {**manifest_body, "manifest_content_sha256": manifest_hash}

if MANIFEST_PATH.exists():
    existing = load_json(MANIFEST_PATH)
    if existing["manifest_content_sha256"] != manifest_hash:
        raise RuntimeError("Existing Stage08 manifest differs")
else:
    with open(MANIFEST_PATH, "x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

print("STAGE 08 DONDURULDU:")
print(MANIFEST_PATH)

STAGE 08 DONDURULDU:
/content/drive/MyDrive/LLM_Confidence_Thesis/features/assembled/validated_v012_final250_v1/stage08_manifest_v1.json


# 08 tamamlanma kriterleri

- [ ] 04/05/06V2/07 manifest SHA zinciri doğrulandı
- [ ] 360 key dört stage arasında birebir eşleşti
- [ ] Frozen answer SHA'ları doğrulandı
- [ ] Reference/aliases/raw answer/token arrays predictor tablosuna alınmadı
- [ ] `manual_error_type` predictor değildir
- [ ] CORE feature listesi explicit olarak donduruldu
- [ ] EXTENDED feature listesi explicit olarak donduruldu
- [ ] Missing / non-finite predictor yok
- [ ] Master Parquet yazıldı
- [ ] Core model matrix yazıldı
- [ ] Extended model matrix yazıldı
- [ ] Feature schema ve Stage08 manifest donduruldu
- [ ] Final test split henüz oluşturulmadı

## Sonraki aşama — 09 Feature Analysis & Baselines

09 development/exploratory analizidir. Model başına:

- prevalence baseline
- correct vs incorrect feature dağılımları
- single-signal AUROC / AUPRC
- signal redundancy/korelasyon
- probability-like sinyaller için Brier/NLL
- reliability ön incelemesi

hesaplanacak. Mevcut 120 soru nihai tez test sonucu olarak raporlanmayacak.